# 153 — AgentOps y análisis de trayectorias

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Éxito: 6/8 = **75 %**. Pasos p50 = **4** (ordenados: 3,3,4,4,5,6,12,12).
Costo medio = 0.53/8 ≈ **$0.066**. Intervención: 2/8 = **25 %**. Patrón de t3 y t5:
ambas con **12 pasos** — exactamente donde el presupuesto parece agotarse — y costo ~3×
la mediana. Hipótesis primera: bucle (el agente repite un paso que falla) con tope de 12
pasos; se confirma leyendo las dos trayectorias y buscando llamadas repetidas con los
mismos argumentos.

**Ejercicio 2.** Éxitos reales = 81 + 3 = **84 %**. Falsos éxitos declarados: 11/92 ≈
**12 %** de lo que el agente afirma haber logrado no ocurrió. El falso éxito es más
peligroso: el falso fallo cuesta un reintento o una revisión (desperdicio visible); el
falso éxito deja al usuario creyendo que el reembolso existe cuando no — un error
silencioso con efecto diferido, la clase de fallo que erosiona la confianza y aparece
semanas después como incidente.

**Ejercicio 3.**

```python
span_herramienta = {
    "tool.name": "crear_reembolso",
    "tool.args.pedido": "9912",
    "tool.args.monto_pct": 100,
    "tool.status": "error",
    "tool.error.tipo": "monto_sobre_limite",
    "trace_id": "T-4471", "paso": 3, "duracion_ms": 210,
}
```

Consulta (pseudocódigo): *agrupar spans de herramienta por `trace_id` donde
`tool.status = error` y los `tool.args` sean idénticos al span anterior de la misma
herramienta; contar trayectorias con ≥ 2 repeticiones*. Eso es literalmente el cluster A:
reintentos con los mismos argumentos tras el mismo error.

**Ejercicio 4.** Los pasos estructurados del resultado juegan el papel de spans (orden,
duración, estado); `evidence` da conteos por paso. Para las cuatro métricas faltan: el
criterio externo de éxito (eval), el costo en tokens/$ por paso y el registro de
intervenciones — exactamente los campos que un runtime de agente real añade a la traza.


In [ ]:
result = run_lab("observability", seed=153)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
import statistics

tareas = [
    {"id": "t1", "exito": True,  "pasos": 3,  "costo": 0.03, "interv": False},
    {"id": "t2", "exito": True,  "pasos": 5,  "costo": 0.05, "interv": False},
    {"id": "t3", "exito": False, "pasos": 12, "costo": 0.14, "interv": True},
    {"id": "t4", "exito": True,  "pasos": 4,  "costo": 0.04, "interv": False},
    {"id": "t5", "exito": False, "pasos": 12, "costo": 0.13, "interv": False},
    {"id": "t6", "exito": True,  "pasos": 6,  "costo": 0.06, "interv": False},
    {"id": "t7", "exito": True,  "pasos": 4,  "costo": 0.05, "interv": True},
    {"id": "t8", "exito": True,  "pasos": 3,  "costo": 0.03, "interv": False},
]
n = len(tareas)
print(f"éxito        = {sum(t['exito'] for t in tareas)/n:.0%}")
print(f"pasos p50    = {statistics.median(t['pasos'] for t in tareas):.0f}")
print(f"costo medio  = ${sum(t['costo'] for t in tareas)/n:.3f}")
print(f"intervención = {sum(t['interv'] for t in tareas)/n:.0%}")
fallidas = [t for t in tareas if not t["exito"]]
print("fallidas:", [(t["id"], t["pasos"]) for t in fallidas], "→ ambas en el tope de pasos: hipótesis de bucle")


## Reflexión

1. ¿Por qué el éxito auto-reportado por el agente sobreestima el éxito real, y qué tres formas de verificación externa podrías usar según el tipo de tarea?
2. En el ejemplo, el arreglo del cluster A fue mejorar el mensaje de error de la herramienta, no el prompt: ¿qué te dice eso sobre dónde vive el «comportamiento» de un agente?
3. ¿Qué condiciones debe cumplir un replay de trayectorias para que la comparación entre la configuración vieja y la nueva sea válida?
